# SILVER (Dados Limpos) 
Dados validados e limpos<br>
Tipos de dados corretos<br>
Valores nulos tratados<br>
Duplicatas removidas<br>

## PROCESSAMENTO DOS DADOS

### IMPORTAÇÃO DAS BIBLIOTECAS

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import datetime
import seaborn as sns
import numpy as np

### CARREGAMENTO DOS DADOS

In [2]:
bronze_path = 'data/bronze/dados_brutos.csv'
df = pd.read_csv(bronze_path)

## TRATAMENTO DOS DADOS


### ALTERAÇÃO DOS TIPOS DE DADOS

In [3]:
# Identificador
df['CODIGO_CLIENTE'] = df['CODIGO_CLIENTE'].astype(str)

# Categóricas 
df['UF'] = df['UF'].astype('category')
df['ESCOLARIDADE'] = df['ESCOLARIDADE'].astype('category')
df['ESTADO_CIVIL'] = df['ESTADO_CIVIL'].astype('category')

# Booleanas 
df['CASA_PROPRIA'] = df['CASA_PROPRIA'].map({'Sim': True, 'Não': False}).astype(np.bool_)
df['OUTRA_RENDA'] = df['OUTRA_RENDA'].map({'Sim': True, 'Não': False}).astype(np.bool_)
df['TRABALHANDO_ATUALMENTE'] = df['TRABALHANDO_ATUALMENTE'].map({'Sim': True, 'Não': False}).astype(np.bool_)

# Numéricas inteiras 
int_cols = ['IDADE', 'QT_FILHOS', 'QT_IMOVEIS', 'TEMPO_ULTIMO_EMPREGO_MESES', 'QT_CARROS', 'SCORE']
for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.int64)

# Valores monetários 
float_cols = ['VL_IMOVEIS', 'OUTRA_RENDA_VALOR', 'ULTIMO_SALARIO', 'VALOR_TABELA_CARROS']
for col in float_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.float64)

### PADRONIZAÇÃO DOS VALORES TEXTUAIS 

In [4]:
text_cols = [c for c in df.columns if df[c].dtype == 'object']
for c in text_cols:
    df[c] = df[c].astype(str).str.strip().str.upper()

### TRATAMENTO DE NULOS

In [5]:
df.isnull().sum()

CODIGO_CLIENTE                0
UF                            0
IDADE                         0
ESCOLARIDADE                  0
ESTADO_CIVIL                  0
QT_FILHOS                     0
CASA_PROPRIA                  0
QT_IMOVEIS                    0
VL_IMOVEIS                    0
OUTRA_RENDA                   0
OUTRA_RENDA_VALOR             0
TEMPO_ULTIMO_EMPREGO_MESES    0
TRABALHANDO_ATUALMENTE        0
ULTIMO_SALARIO                3
QT_CARROS                     0
VALOR_TABELA_CARROS           0
SCORE                         0
DATA_UPLOAD                   0
ARQUIVO_FONTE                 0
dtype: int64

In [6]:
df.replace('SEM DADOS',np.nan, inplace = True)
df['ULTIMO_SALARIO'] = df['ULTIMO_SALARIO'].fillna((df['ULTIMO_SALARIO'].median()))

### TRATAMENTO DE OUTLIERS


In [7]:
moda = df['QT_FILHOS'].mode()[0]
df.loc[df['QT_FILHOS'] > 3, 'QT_FILHOS'] = moda

### TRATAMENTO DE DUPLICATAS

In [8]:
df.duplicated().sum()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10476 entries, 0 to 10475
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   CODIGO_CLIENTE              10476 non-null  object  
 1   UF                          10476 non-null  category
 2   IDADE                       10476 non-null  int64   
 3   ESCOLARIDADE                10476 non-null  category
 4   ESTADO_CIVIL                10476 non-null  category
 5   QT_FILHOS                   10476 non-null  int64   
 6   CASA_PROPRIA                10476 non-null  bool    
 7   QT_IMOVEIS                  10476 non-null  int64   
 8   VL_IMOVEIS                  10476 non-null  float64 
 9   OUTRA_RENDA                 10476 non-null  bool    
 10  OUTRA_RENDA_VALOR           10476 non-null  float64 
 11  TEMPO_ULTIMO_EMPREGO_MESES  10476 non-null  int64   
 12  TRABALHANDO_ATUALMENTE      10476 non-null  bool    
 13  ULTIMO_SALARIO  

### CRIAÇÃO DE COLUNA RENDA_TOTAL

In [9]:
df['RENDA_TOTAL'] = df['ULTIMO_SALARIO'].fillna(0) + df['OUTRA_RENDA_VALOR'].fillna(0)


## SALVAR NA CAMADA SILVER

### ADICIONAR INFORMAÇÃO DE TRATAMENTO DOS DADOS
 

In [19]:
df['DATA_TRATAMENTO'] = datetime.now()

In [20]:
silver_path = 'data/silver/dados_limpos.csv'
df.to_csv(silver_path, index=False)
print(f'\nDados limpos salvos: {silver_path} (shape={df.shape})')


Dados limpos salvos: data/silver/dados_limpos.csv (shape=(10476, 21))
